# CountGD - Multimodal open-world object counting



## Setup

The following cells will setup the runtime environment with the following

- Mount Google Drive
- Install dependencies for running the model
- Load the model into memory

### Mount Google Drive (if running on colab)

The following bit of code will mount your Google Drive folder at `/content/drive`, allowing you to process files directly from it as well as store the results alongside it.

Once you execute the next cell, you will be requested to share access with the notebook. Please follow the instructions on screen to do so.
If you are not running this on colab, you will still be able to use the files available on your environment.

In [ ]:
# Check if running colab
import logging

logging.basicConfig(
  level=logging.INFO,
  format='%(asctime)s %(levelname)-8s %(name)s %(message)s'
)
try:
    import google.colab
    RUNNING_IN_COLAB = True
except:
    RUNNING_IN_COLAB = False

if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

from IPython.core.magic import register_cell_magic
from IPython import get_ipython
@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)


%env RUNNING_IN_COLAB {RUNNING_IN_COLAB}


### Install Dependencies

The environment will be setup with the code, models and required dependencies.

*Note for Colab users*

To reduce the waiting time, you can use the pre-built wheel file available [here](https://drive.google.com/file/d/1Vl_6DAWfnVU7HFX5y_5TqqbkyTcjONbm/view?usp=sharing) - Visit the link and add it as a shortcut to your "My Drive" folder or edit the path accordingly below. (Line 28)

Alternatively, if you are unable to use google drive, you can download the file to your machine & upload it to the colab runtime when you connect to it and update the path below to install it from there. (Line 28)

In [ ]:
%%bash

set -euxo pipefail

if [ "${RUNNING_IN_COLAB}" == "True" ]; then
  echo "Downloading the repository..."
  if [ ! -d /content/countgd ]; then
    git clone "https://huggingface.co/spaces/nikigoli/countgd" /content/countgd
  fi
  cd /content/countgd

  # If you are testing out WIP items, uncomment the following and change the pr ref
  # git fetch origin refs/pr/10:refs/remotes/origin/pr/10
  # git checkout pr/10 && git pull
else
  # TODO check if cwd is the correct git repo
  # If users use vscode, then we set the default start directory to root of the repo
  echo "Running in $(pwd)"
fi

# TODO check for gcc-11 or above

# Install pip packages
pip install --upgrade pip setuptools wheel ninja
pip install -r requirements.txt

cd models/GroundingDINO/ops
if [ "${RUNNING_IN_COLAB}" == "True" ]; then
    export CUDA_HOME=/usr/local/cuda/
    if ! pip install "/content/drive/MyDrive/MultiScaleDeformableAttention-1.0-cp311-cp311-linux_x86_64.whl"
    then
        echo "failed to install wheel, trying to build from source";
        python3 setup.py build
        python3 setup.py install
    fi
else
    # We try to build the module as we dont know what environment we are running on
    python3 setup.py build
    python3 setup.py install
fi
python3 test.py

In [ ]:
%cd {"/content/countgd" if RUNNING_IN_COLAB else '.'}

## Inference

### Loading the model

In [ ]:
import app
import importlib
importlib.reload(app)
from app import (
    build_model_and_transforms,
    get_device,
    get_args_parser,
    generate_heatmap,
    get_xy_from_boxes,
    predict,
)
args = get_args_parser().parse_args([])
device = get_device()
model, transform = build_model_and_transforms(args)
model = model.to(device)

run = lambda image, text: predict(model, transform, image, text, None, device)
get_output = lambda image, boxes: (len(boxes), get_xy_from_boxes(image, boxes), generate_heatmap(image, boxes))


### Input / Output Utils

Helper functions for reading / writing to zipfiles and csv

In [17]:
import io
import csv
from pathlib import Path
from contextlib import contextmanager
import zipfile
import filetype
from PIL import Image
logger = logging.getLogger()

def images_from_zipfile(p: Path):
    if not zipfile.is_zipfile(p):
        raise ValueError(f'{p} is not a zipfile!')

    with zipfile.ZipFile(p, 'r') as zipf:
        def process_entry(info: zipfile.ZipInfo):
            with zipf.open(info) as f:
                if not filetype.is_image(f):
                    logger.debug(f'Skipping file - {info.filename} as it is not an image')
                    return
                # Try loading the file
                try:
                    with Image.open(f) as im:
                        im.load()
                        return (info.filename, im)
                except:
                    logger.exception(f'Error reading file {info.filename}')

        num_files = sum(1 for info in zipf.infolist() if info.is_dir() == False)
        logger.info(f'Found {num_files} file(s) in the zip')
        yield from (process_entry(info) for info in zipf.infolist() if info.is_dir() == False)

@contextmanager
def zipfile_writer(p: Path):
    with zipfile.ZipFile(p, 'w') as zipf:
        def write_output(image, image_filename):
            buf = io.BytesIO()
            image.save(buf, 'PNG')
            zipf.writestr(image_filename, buf.getvalue())
        yield write_output

@contextmanager
def csvfile_writer(p: Path):
    with p.open('w', newline='') as csvfile:
        fieldnames = ['filename', 'count']
        csv_writer = csv.DictWriter(csvfile, fieldnames = fieldnames)
        csv_writer.writeheader()

        yield csv_writer.writerow

In [ ]:
from tqdm import tqdm
import os
import json
def convert_xy_to_json(xy: tuple):
    x, y = xy
    pts = []
    for _x, _y in zip(x.tolist(), y.tolist()):
        _x, _y = round(_x, 3), round(_y, 3)
        pts.append([_x, _y])

    # List of [x, y] points
    return pts

def process_zipfile(input_zipfile: Path, text: str):
    if not input_zipfile.exists() or not input_zipfile.is_file() or not os.access(input_zipfile, os.R_OK):
        logger.error(f'Cannot open / read zipfile: {input_zipfile}. Please check if it exists')
        return

    if text == "":
        logger.error('Please provide the object you would like to count')
        return

    output_zipfile = input_zipfile.parent / f'{input_zipfile.stem}_countgd.zip'
    output_csvfile = input_zipfile.parent / f'{input_zipfile.stem}.csv'
    output_xyjson = input_zipfile.parent / f'{input_zipfile.stem}_xy.json'

    xy_map = {}

    logger.info(f'Writing outputs to {output_zipfile.name} and {output_csvfile.name} in {input_zipfile.parent} folder')
    with zipfile_writer(output_zipfile) as add_to_zip, csvfile_writer(output_csvfile) as write_row:
        for filename, im in tqdm(images_from_zipfile(input_zipfile)):
            try:
                boxes, _ = run(im, text)
                count, xy, heatmap  = get_output(im, boxes)
                logger.info(f'Count: {count} - {filename}')
                xy_map[filename] = convert_xy_to_json(xy)
                write_row({'filename': filename, 'count': count})
                add_to_zip(heatmap, filename)
            except Exception:
                logger.error(f'failed to process {filename}')

    output_xyjson.write_text(json.dumps(xy_map))

### Run

Use the form on colab to set the parameters, providing the zipfile with input images and a promt text representing the object you want to count.

Use the fileupload button to add the zip file to the `countgd` directory or change the path below accordingly.

If you are not running on colab, change the values in the next cell

Make sure to run the cell once you change the value.

In [8]:
# @title ## Parameters { display-mode: "form", run: "auto" }
# @markdown Set the following options to pass to the CountGD Model

# @markdown ---
# @markdown ### Enter a file path to a zip:
zipfile_path = "test_images.zip" # @param {type:"string"}
# @markdown
# @markdown ### Which object would you like to count?
prompt = "strawberry" # @param {type:"string"}
# @markdown ---

In [ ]:
import ipywidgets as widgets
from IPython.display import display
button = widgets.Button(description="Run")

def on_button_clicked(b):
    # Display the message within the output widget.
    process_zipfile(Path(zipfile_path), prompt)

button.on_click(on_button_clicked)
display(button)